# 03 · Gold · `fato_interrupcao`
### V-Energy Grid 360 — Arquitetura Medalhão

Responde à Camila (*"se a rua dele teve falta de luz"*, na visão 360) e à Beatriz (*"saber quais regiões concentram as piores interrupções e onde a manutenção preventiva dá mais retorno"*). Grava `v_energy.gold.fato_interrupcao`, seguindo o **contrato da Gold (tabela A7, versão v4)**, com as 13 colunas do contrato.

**Grão:** 1 linha = 1 ocorrência de interrupção em 1 UC (o glossário do case diz que cada registro é uma UC interrompida). Esperado: **180.000 linhas**, com `id_ocorrencia` único.

**Quem usa:** visão 360 (a UC ficou sem luz?), tela de qualidade (interrupções por causa e local), a `dec_fec_alimentador_mensal` (A8) e o agente de IA ("falta de luz").

**Fontes (Silver):** `ocorrencias_rede` (já tratada: causa ausente vira nulo, duração numérica, datas com horário; log O1 a O3) e `unidades_consumidoras` (município, bairro e classe da UC).

| # | Passo | Regra |
|---|---|---|
| 1 | Ocorrência | `silver.ocorrencias_rede`, sem remover linhas |
| 2 | Localização da UC | `municipio`, `bairro` e `classe` de `silver.unidades_consumidoras` (`left join`) |
| 3 | Datas | `data_abertura` e `data_encerramento` viram `data_hora_inicio` e `data_hora_fim`, **com horário** (sem cast para `date`) |
| 4 | `causa_evitavel` | `true` para `VEGETACAO`, `FALHA DE EQUIPAMENTO` e `SOBRECARGA`; `false` para as demais **e para a causa nula** |

**Decisões deste notebook:**
- **Causa nula não é evitável.** As 5.400 ocorrências sem causa (log O1) ficam com `causa_evitavel = false`, e não nulo: sem saber a causa, não dá para dizer que a manutenção evitaria. Assim, `horas_causas_evitaveis` da A8 é um **piso**. `NAO IDENTIFICADA` também é `false`.
- **`OSCILACAO` entra** (14.562 ocorrências): o glossário diz que cada registro é uma UC interrompida. É uma suposição a registrar no README; quem quiser excluir filtra por `tipo`.
- **"A rua dele teve falta de luz":** esta tabela diz se **a própria UC** ficou sem luz. Para a rua, o backend busca as ocorrências do mesmo alimentador ou do mesmo logradouro e bairro no período (contrato).
- **Sem tabela temporária (`_stg`):** são 180 mil linhas e um único `join` com 45 mil UCs; recalcular a cada checagem é rápido.

**Pré-requisito:** `silver.ocorrencias_rede` e `silver.unidades_consumidoras` gravadas. Para o teste de conexão, a `gold.dim_uc`.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row
from datetime import datetime

tabela_oc = f"{silver_schema}.ocorrencias_rede"
tabela_uc = f"{silver_schema}.unidades_consumidoras"
tabela_gold = f"{gold_schema}.fato_interrupcao"
tabela_dim = f"{gold_schema}.dim_uc"                # só para o teste de conexão (seção 9)
tabela_dq_log = f"{silver_schema}.dq_log"           # mesmo log das Silvers e das outras Golds

CAUSAS_EVITAVEIS = ["VEGETACAO", "FALHA DE EQUIPAMENTO", "SOBRECARGA"]
MES_INI, MES_FIM = "2022-07", "2026-06"

def n_br(n):
    return f"{n:,}".replace(",", ".")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_oc}, {tabela_uc}")
print(f"destino: {tabela_gold}")

## 2. Funções de checagem de qualidade

Mesmas funções das Silvers e das outras Golds (`dq_check`, `dq_check_unique`, `dq_check_contagem` e `dq_check_valor`), com as mesmas colunas no log. Os resultados vão para `silver.dq_log` (sempre), e a tabela só é gravada se **todas** as checagens passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=dupes, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=diferenca, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado (soma que precisa bater com a Silver ou com o log)."""
    passed = obtido == esperado
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
                          failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 3. Leitura das Silvers e conferência das colunas

Antes de construir, o notebook confere que as Silvers existem **com as colunas de que precisa**; se faltar alguma, para aqui com uma mensagem clara.

Contagens esperadas: `silver.ocorrencias_rede` com **180.000** linhas e `silver.unidades_consumidoras` com **45.000**.

In [0]:
oc = spark.table(tabela_oc)
uc = spark.table(tabela_uc)

exigidas = {
    tabela_oc: ["id_ocorrencia", "id_uc", "alimentador", "tipo", "causa", "data_abertura", "data_encerramento",
                "mes_referencia", "duracao_min"],
    tabela_uc: ["id_uc", "municipio", "bairro", "classe", "alimentador"],
}
for tabela, colunas in exigidas.items():
    faltando = [c for c in colunas if c not in spark.table(tabela).columns]
    if faltando:
        raise Exception(f"{tabela} não tem as colunas {faltando}. Ajuste os nomes antes de seguir (DESCRIBE {tabela}).")

n_oc, n_uc = oc.count(), uc.count()
print(f"ocorrências: {n_br(n_oc)} | UCs: {n_br(n_uc)}")   # esperado: 180.000 | 45.000
oc.printSchema()

## 4. Montagem da `fato_interrupcao`

**Decisão:** partir de `silver.ocorrencias_rede` e acrescentar, com `left join` (nenhuma ocorrência se perde), o município, o bairro e a classe da UC. Se alguma UC faltasse no cadastro, a linha ficaria com nulo e a checagem da seção 5 acusaria.

**Alimentador:** vem da ocorrência. A Silver já garante que ele é **sempre igual** ao do cadastro da UC (checagem da Silver), e a seção 5 confere de novo.

**`causa_evitavel`:** `true` só para `VEGETACAO`, `FALHA DE EQUIPAMENTO` e `SOBRECARGA`; `false` para `CLIMATICA`, `TERCEIROS`, `NAO IDENTIFICADA` **e para a causa nula** (5.400). Com `coalesce`, nunca fica nula.

**Ordem das colunas:** a do contrato (13 colunas).

**Conferência:** 180.000 linhas, 0 sem município, bairro ou classe.

In [0]:
uc_sel = uc.select("id_uc", "municipio", "bairro", "classe", F.col("alimentador").alias("alimentador_cadastro"))

base = (
    oc.join(F.broadcast(uc_sel), "id_uc", "left")
    .withColumn("causa_evitavel", F.coalesce(F.col("causa").isin(*CAUSAS_EVITAVEIS), F.lit(False)))
)

df_fato = base.select(
    "id_ocorrencia", "id_uc", "alimentador", "municipio", "bairro", "classe",
    "tipo", "causa", "causa_evitavel",
    F.col("data_abertura").alias("data_hora_inicio"),
    F.col("data_encerramento").alias("data_hora_fim"),
    "mes_referencia",
    F.col("duracao_min").cast("int").alias("duracao_min"),
)
print("colunas:", len(df_fato.columns))   # esperado: 13
df_fato.printSchema()
print("linhas:", n_br(df_fato.count()))   # esperado: 180.000
print("sem município, bairro ou classe:", n_br(df_fato.filter("municipio IS NULL OR bairro IS NULL OR classe IS NULL").count()))   # esperado: 0
display(df_fato.groupBy("causa", "causa_evitavel").count().orderBy("causa_evitavel", "causa"))

## 5. Checagens de qualidade

| Checagem | Protege |
|---|---|
| Total = 180.000; `id_ocorrencia` único; igual à Silver | `join` que perde ou multiplica linhas |
| Colunas obrigatórias não nulas (`causa` pode ser nula) | junção sem correspondência |
| `id_ocorrencia`, `id_uc` e `alimentador` no padrão | formato das chaves |
| Alimentador igual ao do cadastro da UC | o DEC/FEC por alimentador (A8) não depende de qual fonte usar |
| `tipo` nos 5 valores; `causa` nos 6 ou nula; `classe` nas 5 | domínios |
| **`causa_evitavel` coerente com a causa** (inclui causa nula = `false`) | a regra da manutenção preventiva |
| **5.400 causas nulas** (log O1) | o tratamento da Silver chegou até aqui |
| `fim − início = duracao_min` (tolerância de 1 min); fim não antes do início | coerência das datas com a duração (base do DEC) |
| `mes_referencia` = mês do início, entre `2022-07` e `2026-06` | derivado e período |
| **Soma de `duracao_min` = soma da Silver** | a Gold não alterou nenhuma duração |
| **122 alimentadores** e **17.844 ocorrências em `2025-07`** (evento climático; log A6) | números conhecidos do case e do diagnóstico |

Os valores esperados valem para esta extração; numa reexecução com dados novos, precisam ser atualizados.

In [0]:
t = "gold.fato_interrupcao"
dq_results.clear()   # se esta célula for reexecutada, não acumula resultados antigos

TIPOS = ["FALTA DE ENERGIA", "DANO EM EQUIPAMENTO", "OSCILACAO", "SOBRECARGA", "FIO PARTIDO"]
CAUSAS = ["CLIMATICA", "SOBRECARGA", "NAO IDENTIFICADA", "VEGETACAO", "TERCEIROS", "FALHA DE EQUIPAMENTO"]
CLASSES = ["RESIDENCIAL", "RURAL", "COMERCIAL", "PODER PUBLICO", "INDUSTRIAL"]

def ok(cond):
    return F.coalesce(cond, F.lit(False))   # nulo também falha

fato = df_fato
n_fato = fato.count()

# Volume e chave
dq_check_contagem(t, "total de linhas = 180.000", fato, 180_000)
dq_check_unique(t, "id_ocorrencia único", fato, ["id_ocorrencia"])
dq_check_valor(t, "linhas = silver.ocorrencias_rede", n_fato, n_fato, n_oc)

# Nulos (causa pode ser nula: 5.400 sem causa informada)
for coluna in ["id_ocorrencia", "id_uc", "alimentador", "municipio", "bairro", "classe", "tipo", "causa_evitavel",
               "data_hora_inicio", "data_hora_fim", "mes_referencia", "duracao_min"]:
    dq_check(t, f"{coluna} não nulo", fato, F.col(coluna).isNotNull())

# Formato das chaves e domínios
dq_check(t, "id_uc no padrão UC + 6 dígitos", fato, ok(F.col("id_uc").rlike("^UC[0-9]{6}$")))
dq_check(t, "alimentador no padrão ALM-XXX-000", fato, ok(F.col("alimentador").rlike("^ALM-[A-Z]{3}-[0-9]{3}$")))
dq_check(t, "tipo nos 5 valores", fato, ok(F.col("tipo").isin(*TIPOS)))
dq_check(t, "causa nos 6 valores ou nula", fato, F.col("causa").isNull() | ok(F.col("causa").isin(*CAUSAS)))
dq_check(t, "classe nas 5 classes", fato, ok(F.col("classe").isin(*CLASSES)))

# Alimentador da ocorrência = alimentador do cadastro
dq_check(t, "alimentador igual ao do cadastro da UC", base, ok(F.col("alimentador") == F.col("alimentador_cadastro")))

# causa_evitavel
dq_check(t, "causa_evitavel = true só nas 3 causas evitáveis", fato,
         F.col("causa_evitavel") == ok(F.col("causa").isin(*CAUSAS_EVITAVEIS)))
dq_check_valor(t, "causas nulas = 5.400 (log O1)", n_fato, fato.filter("causa IS NULL").count(), 5_400)

# Datas e duração
dif_min = (F.unix_timestamp("data_hora_fim") - F.unix_timestamp("data_hora_inicio")) / 60
dq_check(t, "fim - início = duracao_min (tolerância de 1 min)", fato, ok(F.abs(dif_min - F.col("duracao_min")) <= 1))
dq_check(t, "fim não antes do início", fato, ok(F.col("data_hora_fim") >= F.col("data_hora_inicio")))
dq_check(t, "duracao_min > 0", fato, ok(F.col("duracao_min") > 0))
dq_check(t, "mes_referencia = mês do início", fato, ok(F.col("mes_referencia") == F.date_format("data_hora_inicio", "yyyy-MM")))
dq_check(t, f"mes_referencia entre {MES_INI} e {MES_FIM}", fato, ok(F.col("mes_referencia").between(MES_INI, MES_FIM)))
dq_check_valor(t, "datas com horário (há inícios fora da meia-noite)", n_fato,
               fato.filter("hour(data_hora_inicio) <> 0 OR minute(data_hora_inicio) <> 0").count() > 0, True)

# Reconciliação com a Silver e números conhecidos
dq_check_valor(t, "soma de duracao_min = silver.ocorrencias_rede", n_fato,
               fato.agg(F.sum("duracao_min")).first()[0], oc.agg(F.sum(F.col("duracao_min").cast("long"))).first()[0])
dq_check_valor(t, "alimentadores distintos = 122", n_fato, fato.select("alimentador").distinct().count(), 122)
dq_check_valor(t, "ocorrências em 2025-07 = 17.844 (evento climático)", n_fato,
               fato.filter("mes_referencia = '2025-07'").count(), 17_844)

## 6. Registro das checagens em `silver.dq_log`

Gravado **sempre**, mesmo com falha (modo `append`), com `table_name = gold.fato_interrupcao`.

In [0]:
spark.createDataFrame(dq_results).write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(tabela_dq_log)
display(spark.table(tabela_dq_log).filter(F.col("table_name") == t)
        .orderBy(F.col("checked_at").desc()).limit(len(dq_results)))

## 7. Gravação da Gold, só se todas as checagens passarem

In [0]:
falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

df_fato.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_gold)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

## 8. Teste de conexão e consultas de conferência

1. **Conexão com a `dim_uc`:** toda UC com interrupção existe na `gold.dim_uc`, com o mesmo município, bairro, classe e alimentador.
2. **Visão 360 (Camila):** as interrupções de uma UC, da mais recente para a mais antiga.
3. **Por causa (Beatriz):** ocorrências, horas e duração mediana por causa; o peso das evitáveis.
4. **Por região (Beatriz, P9):** horas interrompidas por município e as horas evitáveis, com o número de UCs para dar a escala.
5. **No tempo:** ocorrências por ano e o evento de julho de 2025.
6. **Agente (P10):** exemplos de pergunta sobre "falta de luz".

In [0]:
f = spark.table(tabela_gold)

# 1. Conexão com a dim_uc (só roda se a gold.dim_uc existir neste workspace)
if spark.catalog.tableExists(tabela_dim):
    d = spark.table(tabela_dim).select("id_uc", F.col("municipio").alias("m_d"), F.col("bairro").alias("b_d"),
                                       F.col("classe").alias("c_d"), F.col("alimentador").alias("a_d"))
    ucs_f = f.select("id_uc", "municipio", "bairro", "classe", "alimentador").distinct()
    print("UCs com interrupção fora da gold.dim_uc:", n_br(ucs_f.join(d, "id_uc", "left_anti").count()))   # esperado: 0
    dif = ucs_f.join(d, "id_uc").filter(~(F.col("municipio").eqNullSafe(F.col("m_d")) & F.col("bairro").eqNullSafe(F.col("b_d"))
                                          & F.col("classe").eqNullSafe(F.col("c_d")) & F.col("alimentador").eqNullSafe(F.col("a_d"))))
    print("UCs com município, bairro, classe ou alimentador diferente:", n_br(dif.count()))                 # esperado: 0
    print("UCs com pelo menos uma interrupção:", n_br(ucs_f.select("id_uc").distinct().count()), "de 45.000")
else:
    print(f"{tabela_dim} não existe neste workspace: teste de conexão fica para quem tiver as duas tabelas")

# 2. Visão 360: as interrupções da UC que mais teve
uc_exemplo = f.groupBy("id_uc").count().orderBy(F.desc("count"), "id_uc").first()["id_uc"]
print("UC de exemplo:", uc_exemplo)
display(f.filter(F.col("id_uc") == uc_exemplo)
        .select("data_hora_inicio", "data_hora_fim", "duracao_min", "tipo", "causa", "causa_evitavel", "alimentador")
        .orderBy(F.desc("data_hora_inicio")))

In [0]:
# 3. Por causa: ocorrências, horas e duração mediana (esperado: CLIMATICA ~36% das horas, 116.919 h, mediana 101 min)
tot_h = f.agg(F.sum("duracao_min")).first()[0] / 60
display(f.groupBy(F.coalesce("causa", F.lit("(sem causa)")).alias("causa"), "causa_evitavel")
        .agg(F.count("*").alias("ocorrencias"),
             F.round(F.sum("duracao_min") / 60, 0).alias("horas"),
             F.round(F.sum("duracao_min") / 60 / F.lit(tot_h) * 100, 1).alias("pct_horas"),
             F.expr("percentile_approx(duracao_min, 0.5)").alias("duracao_mediana_min"))
        .orderBy(F.desc("horas")))

ev = f.agg(F.sum(F.when(F.col("causa_evitavel"), F.col("duracao_min"))).alias("ev"), F.sum("duracao_min").alias("tot")).first()
print(f"horas totais: {n_br(round(ev['tot'] / 60))} | horas de causas evitáveis: {n_br(round(ev['ev'] / 60))} ({ev['ev'] / ev['tot'] * 100:.1f}%)")

# Por tipo (OSCILACAO entra no DEC/FEC: suposição a registrar no README)
display(f.groupBy("tipo").agg(F.count("*").alias("ocorrencias"), F.round(F.sum("duracao_min") / 60, 0).alias("horas")).orderBy(F.desc("ocorrencias")))

In [0]:
# 4. Por região: horas por município, com o número de UCs e as horas por UC (escala), e as horas evitáveis
ucs_mun = uc.groupBy("municipio").agg(F.count("*").alias("ucs"))
display(f.groupBy("municipio")
        .agg(F.count("*").alias("ocorrencias"),
             F.round(F.sum("duracao_min") / 60, 0).alias("horas"),
             F.round(F.sum(F.when(F.col("causa_evitavel"), F.col("duracao_min"))) / 60, 0).alias("horas_evitaveis"))
        .join(ucs_mun, "municipio")
        .withColumn("horas_por_uc", F.round(F.col("horas") / F.col("ucs"), 2))
        .withColumn("pct_evitavel", F.round(F.col("horas_evitaveis") / F.col("horas") * 100, 1))
        .orderBy(F.desc("horas")))

# Os 10 bairros com mais horas evitáveis por UC (bairro sempre com o município; denominador = UCs do bairro)
ucs_bairro = uc.groupBy("municipio", "bairro").agg(F.count("*").alias("ucs"))
display(f.filter("causa_evitavel").groupBy("municipio", "bairro")
        .agg(F.count("*").alias("ocorrencias_evitaveis"), F.round(F.sum("duracao_min") / 60, 0).alias("horas_evitaveis"))
        .join(ucs_bairro, ["municipio", "bairro"])
        .withColumn("horas_evitaveis_por_uc", F.round(F.col("horas_evitaveis") / F.col("ucs"), 2))
        .filter("ucs >= 50")
        .orderBy(F.desc("horas_evitaveis_por_uc")).limit(10))

In [0]:
# 5. No tempo (esperado: 2025 com ~41% mais ocorrências que 2023 e 2024; 2025-07 com 17.844, 97% climática)
display(f.groupBy(F.substring("mes_referencia", 1, 4).alias("ano")).agg(F.count("*").alias("ocorrencias"),
        F.round(F.sum("duracao_min") / 60, 0).alias("horas")).orderBy("ano"))
display(f.filter("mes_referencia = '2025-07'").groupBy("causa").count().orderBy(F.desc("count")))
display(f.filter("mes_referencia = '2025-07'").groupBy("alimentador").count().orderBy(F.desc("count")).limit(5))

# 6. Agente (P10): "quantas vezes faltou luz em Jaboatão em 2025?" e "qual a causa mais comum de falta de luz em Olinda?"
print("Interrupções em Jaboatão dos Guararapes em 2025:",
      n_br(f.filter("municipio = 'Jaboatão dos Guararapes' AND mes_referencia LIKE '2025-%'").count()))
display(f.filter("municipio = 'Olinda'").groupBy("causa").count().orderBy(F.desc("count")).limit(3))